## 0. Entorno

In [76]:
import pandas as pd
import openpyxl
import numpy as np
import seaborn as sns
import matplotlib as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path
import json
import requests
import time
import mysql.connector

#Sirve para cargar el .env con la credenciales de la AEMET
from dotenv import load_dotenv
import os

In [40]:
import os
os.getcwd()

'/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-'

## 1. Exploración de datos

In [41]:
#carga CSV 'FAOSTAT'
agric_ONU = pd.read_csv('/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/FAOSTAT_data_es_6-11-2026.csv')

In [42]:
#carga CSV 'NASA'
clima_NASA = pd.read_csv(
    '/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/POWER_Point_Monthly_20050101_20241231_040d42N_003d70W_LST.csv',
    skiprows=13,
    na_values=['-999', '-999.0']
)

In [60]:
#carga de fichero .xlsx 'ESYRCE'
agriC_ESP = pd.read_excel(r'/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/Dinaweb2024-2.xlsx')

In [104]:
#Carga de datos de la API de la AEMET
#Carga de la API Key

load_dotenv('apikey.env')
API_KEY = os.getenv("AEMET_API_KEY")

#Creación de diccionario que relaciona cada estación de la AEMET con el cultivo que me interesa
estaciones = {
    "cereales": "2422",
    "vinedo": "9145Y",
    "olivar": "5298X",
    "citricos": "8072Y"
}

In [ ]:
#Función para descargar json
def descargar_json_aemet(
    api_key,
    estacion,
    fecha_ini,
    fecha_fin,
    fichero_salida
):  
    endpoint = (
        f"https://opendata.aemet.es/opendata/api/"
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{fecha_ini}T00:00:00UTC/"
        f"fechafin/{fecha_fin}T23:59:59UTC/"
        f"estacion/{estacion}"
    )

    headers = {
        "api_key": api_key
    }

    respuesta = requests.get(
        endpoint,
        headers=headers
    )

    metadata = respuesta.json()
   
    print(metadata)

    if metadata["estado"] != 200:
        print(metadata)
        return

    datos_url = metadata["datos"]

    datos = requests.get(datos_url).json()

    respuesta_datos = requests.get(datos_url)

    if respuesta_datos.status_code != 200:
     print(print(
        f"Fallo en {estacion} "
        f"{fecha_ini} - {fecha_fin}"
        ))
     return

    try:
        datos = respuesta_datos.json()

    except Exception:
        print(
        f"No se pudo leer JSON: "
        f"{fecha_ini} - {fecha_fin}"
     )
        return

    with open(
        fichero_salida,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            datos,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(
        f"Guardado {fichero_salida}"
    )

In [ ]:
for anio in range(2004, 2024):

    PAUSA = 20

    descargar_json_aemet(
        API_KEY,
        "2422",
        f"{anio}-01-01",
        f"{anio}-06-30",
        f"data/AEMET/valladolid/valladolid_{anio}_S1.json"
    )

    time.sleep(PAUSA)

    descargar_json_aemet(
        API_KEY,
        "2422",
        f"{anio}-07-01",
        f"{anio}-12-31",
        f"data/AEMET7valladolid/valladolid_{anio}_S2.json"
    )

    time.sleep(PAUSA)

print("Descarga finalizada")

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/8774e1c7', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2004_S1.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/ad36dfe4', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2004_S2.json
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/851a5907', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2005_S2.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/7381d001', 'metadatos': 'https:/

In [133]:
#Descarga solo de un semestre
descargar_json_aemet(
    API_KEY,
    "2422",
    "2024-07-01",
    "2024-12-31",
    "data/AEMET/valladolid/valladolid_2024_S1.json"
)

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/6c9a680e', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid/valladolid_2024_S1.json


In [103]:
for anio in range(2004, 2025):

    endpoint = (
        f"https://opendata.aemet.es/opendata/api/"
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{anio}-01-01T00:00:00UTC/"
        f"fechafin/{anio}-06-30T23:59:59UTC/"
        f"estacion/2422"
    )

    respuesta = requests.get(
        endpoint,
        headers={"api_key": API_KEY}
    )

    metadata = respuesta.json()

    if metadata["estado"] == 200:
        print(f"✅ {anio}")
    else:
        print(f"❌ {anio}")

✅ 2004
✅ 2005
✅ 2006
✅ 2007
✅ 2008
✅ 2009
✅ 2010
✅ 2011
✅ 2012
✅ 2013
✅ 2014
✅ 2015
✅ 2016
✅ 2017
✅ 2018
✅ 2019
✅ 2020
✅ 2021
✅ 2022
✅ 2023
✅ 2024
